# تحسين جودة الصورة وإزالة النغمشة (Real-ESRGAN)

الكود ده بيعمل:
1. رفع صورة من جهازك
2. تكبير ورفع جودتها (Super Resolution) لحد **x4**
3. إزالة النويز/النغمشة (Denoise)
4. تنزيل الصورة الناتجة بجودة عالية جدًا

> شغّل كل خلية بالترتيب (Shift + Enter). أول مرة هتاخد شوية وقت عشان بيثبت المكتبات.


In [ ]:
# الخطوة 1: تثبيت المكتبات المطلوبة
!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan
!pip install -q opencv-python-headless


In [ ]:
# الخطوة 2: إصلاح مشكلة توافق شائعة بين basicsr و torchvision الحديثة
import os, sys

basicsr_degradations_path = None
for p in sys.path:
    candidate = os.path.join(p, "basicsr", "data", "degradations.py")
    if os.path.exists(candidate):
        basicsr_degradations_path = candidate
        break

if basicsr_degradations_path:
    with open(basicsr_degradations_path, "r") as f:
        content = f.read()
    content = content.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale"
    )
    with open(basicsr_degradations_path, "w") as f:
        f.write(content)
    print("تم إصلاح ملف basicsr بنجاح ✅")
else:
    print("متلقيتش الملف، هنكمل عادي وممكن الخطوة الجاية تشتغل برضه.")


In [ ]:
# الخطوة 3: تحميل موديل Real-ESRGAN (أفضل موديل لرفع الجودة x4 وإزالة النغمشة)
import os

model_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
model_path = "RealESRGAN_x4plus.pth"

if not os.path.exists(model_path):
    !wget -q {model_url} -O {model_path}

print("الموديل جاهز ✅")


In [ ]:
# الخطوة 4: رفع الصورة من جهازك
from google.colab import files

uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f"تم رفع الصورة: {input_filename}")


In [ ]:
# الخطوة 5: تشغيل التحسين (رفع الجودة + إزالة النغمشة)
import cv2
import torch
import numpy as np
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# تحميل بنية الموديل
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("تنبيه: مفيش GPU متاح دلوقتي. اذهب لـ Runtime > Change runtime type > GPU عشان السرعة تبقى أحسن بكتير.")

upsampler = RealESRGANer(
    scale=4,
    model_path="RealESRGAN_x4plus.pth",
    model=model,
    tile=256,          # قلل الرقم ده لو حصل خطأ نقص ذاكرة (مثلاً 128)
    tile_pad=10,
    pre_pad=0,
    half=(device == "cuda")
)

# قراءة الصورة
img = cv2.imread(input_filename, cv2.IMREAD_COLOR)

# خطوة إضافية لإزالة النغمشة قبل الرفع (Denoise)
img_denoised = cv2.fastNlMeansDenoisingColored(img, None, h=7, hColor=7, templateWindowSize=7, searchWindowSize=21)

# رفع الجودة x4 مع الموديل
output, _ = upsampler.enhance(img_denoised, outscale=4)

output_filename = "output_enhanced_" + input_filename
cv2.imwrite(output_filename, output)

print(f"تمت المعالجة بنجاح ✅ الصورة اتحفظت باسم: {output_filename}")
print(f"الأبعاد الأصلية: {img.shape[1]}x{img.shape[0]}  ->  الأبعاد الجديدة: {output.shape[1]}x{output.shape[0]}")


In [ ]:
# الخطوة 6: تنزيل الصورة النهائية على جهازك
from google.colab import files
files.download(output_filename)


### ملاحظات مهمة
- لو الصورة كبيرة أو حصل **Out of Memory**: قلل قيمة `tile` في الخطوة 5 (مثلاً من 256 لـ 128 أو 64).
- لتحسين السرعة بشكل كبير: من قائمة Colab اختار **Runtime > Change runtime type > T4 GPU** قبل ما تشغل الخلايا.
- لو عايز رفع أقل من x4 (مثلاً x2) غيّر قيمة `outscale=4` في الخطوة 5.
- الموديل ده (RealESRGAN_x4plus) مناسب للصور العادية (بورتريه، مناظر، منتجات). لو الصور رسوم/أنمي في نسخة تانية اسمها RealESRGAN_x4plus_anime_6B ممكن أجيبها لو حابب.
